# Analyze Dumped Model Inputs

这个 notebook 用来分析 `train_light.py` 训练时导出的模型输入 dump。

默认读取 `weights_japan_overfit/model_input_dumps`，可按需要修改 `DUMP_ROOT`。

In [ ]:
from pathlib import Path
import json
import math
import torch
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['axes.grid'] = True

DUMP_ROOT = Path('weights_japan_overfit/model_input_dumps')
assert DUMP_ROOT.exists(), f'Dump dir not found: {DUMP_ROOT.resolve()}'

manifest_path = DUMP_ROOT / 'manifest.json'
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
manifest

In [ ]:
def list_epochs(split='train'):
    split_dir = DUMP_ROOT / split
    if not split_dir.exists():
        return []
    return sorted([p.name for p in split_dir.iterdir() if p.is_dir()])

def list_batches(split='train', epoch='epoch_001'):
    epoch_dir = DUMP_ROOT / split / epoch
    if not epoch_dir.exists():
        return []
    return sorted([p.name for p in epoch_dir.glob('batch_*.pt')])

def load_batch(split='train', epoch='epoch_001', batch='batch_0000.pt'):
    path = DUMP_ROOT / split / epoch / batch
    return torch.load(path, map_location='cpu')

print('train epochs:', list_epochs('train'))
print('val epochs:', list_epochs('val'))

In [ ]:
batch = load_batch('train', list_epochs('train')[0], list_batches('train', list_epochs('train')[0])[0])
print(batch.keys())
print('epoch:', batch['epoch'], 'split:', batch['split'], 'batch_idx:', batch['batch_idx'])
print('input tensor shapes:')
for i, x in enumerate(batch['inputs']):
    if torch.is_tensor(x):
        print(f'  inputs[{i}] ->', tuple(x.shape), x.dtype)
    else:
        print(f'  inputs[{i}] ->', type(x))
print('label tensor shapes:')
for i, x in enumerate(batch['labels']):
    print(f'  labels[{i}] ->', tuple(x.shape), x.dtype)
print('p_pick_info keys:', sorted(batch['p_pick_info'].keys()))

In [ ]:
def summarize_batch(batch):
    wave = batch['inputs'][0]             # (B, S, C, T)
    metadata = batch['inputs'][1]         # (B, S, D)
    station_valid = batch['inputs'][2]    # (B, S)
    p_pick_info = batch['p_pick_info']
    print('wave mean/std:', float(wave.mean()), float(wave.std()))
    print('metadata abs mean/max:', float(metadata.abs().mean()), float(metadata.abs().max()))
    print('valid stations per sample:', station_valid.sum(dim=1).tolist())
    if 'event_id' in p_pick_info:
        print('event_id:', p_pick_info['event_id'])
    if 'selected_input_indices' in p_pick_info:
        print('selected_input_indices[0]:', p_pick_info['selected_input_indices'][0].tolist())
    if len(batch['labels']) >= 3:
        pga = batch['labels'][-1]
        print('pga label mean/std:', float(pga.mean()), float(pga.std()))

summarize_batch(batch)

In [ ]:
def plot_waveforms(batch, sample_idx=0, station_idx=0, channels=(0, 1, 2), show_raw=True):
    inputs = batch['inputs']
    pinfo = batch['p_pick_info']
    wave = inputs[0][sample_idx, station_idx].numpy()  # (C, T)
    shifted_p = float(pinfo['shifted'][sample_idx, station_idx])
    raw_p = float(pinfo['raw'][sample_idx, station_idx])
    ncols = 2 if show_raw and 'debug_raw_waveforms' in pinfo else 1
    fig, axes = plt.subplots(len(channels), ncols, figsize=(6 * ncols, 2.5 * len(channels)), squeeze=False)
    for r, ch in enumerate(channels):
        ax = axes[r, 0]
        ax.plot(wave[ch], lw=1)
        ax.axvline(shifted_p, color='tab:red', ls='--', label='shifted p_pick')
        ax.set_title(f'model input | sample={sample_idx} station={station_idx} ch={ch}')
        ax.legend(loc='upper right')
        if ncols == 2:
            raw_wave = pinfo['debug_raw_waveforms'][sample_idx, station_idx].numpy()
            ax2 = axes[r, 1]
            ax2.plot(raw_wave[ch], lw=1)
            ax2.axvline(raw_p, color='tab:orange', ls='--', label='raw p_pick')
            ax2.set_title(f'preprocess snapshot | sample={sample_idx} station={station_idx} ch={ch}')
            ax2.legend(loc='upper right')
    plt.tight_layout()

plot_waveforms(batch, sample_idx=0, station_idx=0)

In [ ]:
def plot_coords(batch, sample_idx=0):
    inputs = batch['inputs']
    pinfo = batch['p_pick_info']
    coords = inputs[1][sample_idx].numpy()
    valid = inputs[2][sample_idx].numpy().astype(bool)
    fig, axes = plt.subplots(1, 2 if 'debug_raw_metadata' in pinfo else 1, figsize=(12, 5))
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])
    axes[0].scatter(coords[valid, 1], coords[valid, 0], c='tab:blue', label='model coords')
    axes[0].set_title('Model input coords')
    axes[0].set_xlabel('coord[1]')
    axes[0].set_ylabel('coord[0]')
    axes[0].legend()
    if 'debug_raw_metadata' in pinfo:
        raw_coords = pinfo['debug_raw_metadata'][sample_idx].numpy()
        raw_valid = pinfo['debug_raw_station_valid'][sample_idx].numpy().astype(bool)
        axes[1].scatter(raw_coords[raw_valid, 1], raw_coords[raw_valid, 0], c='tab:green', label='raw coords')
        axes[1].set_title('Preprocess snapshot coords')
        axes[1].set_xlabel('lon / coord[1]')
        axes[1].set_ylabel('lat / coord[0]')
        axes[1].legend()
    plt.tight_layout()

plot_coords(batch, sample_idx=0)

In [ ]:
def plot_pick_and_pga_distributions(batch, sample_idx=0):
    pinfo = batch['p_pick_info']
    shifted = pinfo['shifted'][sample_idx].numpy()
    raw = pinfo['raw'][sample_idx].numpy()
    valid_shifted = shifted[shifted > 0]
    valid_raw = raw[raw > 0]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    if len(valid_raw) > 0:
        axes[0].hist(valid_raw, bins=20, color='tab:orange', alpha=0.8)
    axes[0].set_title('Raw p_pick distribution')
    if len(valid_shifted) > 0:
        axes[1].hist(valid_shifted, bins=20, color='tab:red', alpha=0.8)
    axes[1].set_title('Shifted p_pick distribution')
    if len(batch['labels']) >= 3:
        pga = batch['labels'][-1][sample_idx, :, 0].numpy()
        axes[2].hist(pga, bins=20, color='tab:blue', alpha=0.8)
        axes[2].set_title('PGA label distribution')
    plt.tight_layout()

plot_pick_and_pga_distributions(batch, sample_idx=0)